## Basic RAG

In [10]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableLambda




### Load tHe docuents 

In [2]:
loader = TextLoader("data/leave_policy.txt",encoding="utf-8")
raw_docs = loader.load()


### split the documenst 

In [3]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=30
)

chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'data/leave_policy.txt'}, page_content='Full-time employees receive 12 sick-leave days per calendar year.\nUnused sick leave is handled according to company policy.\n\nEmployees can request annual leave through the HR portal.\nThe manager must approve the request.'),
 Document(metadata={'source': 'data/leave_policy.txt'}, page_content='Employees must submit travel reimbursement claims within 30 days\nof completing the business trip.')]

## Embeddig Model 

In [4]:
# 3. Create embeddings using a local Hugging Face model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\ladvi\Software-Testing\GIT Repos\langchain-learning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5353.32it/s]


### Ceate a evctore store

In [5]:
vector_store = FAISS.from_documents(
    embedding=embeddings,
    documents=chunks
)


## Create a Retriver

In [6]:
retriever = vector_store.as_retriever(
search_kwargs={"k": 3}
)

### user question 

In [7]:
# 6. Search the documents
question = "How many sick-leave days do employees receive?"

### reteieverd documenst 

In [8]:
retrieved_docs = retriever.invoke(question)
for index, doc in enumerate(retrieved_docs, start=1):
    print(f"\nChunk {index}: {doc.page_content}")
    print("Metadata:", doc.metadata)


Chunk 1: Full-time employees receive 12 sick-leave days per calendar year.
Unused sick leave is handled according to company policy.

Employees can request annual leave through the HR portal.
The manager must approve the request.
Metadata: {'source': 'data/leave_policy.txt'}

Chunk 2: Employees must submit travel reimbursement claims within 30 days
of completing the business trip.
Metadata: {'source': 'data/leave_policy.txt'}


In [9]:
retrieved_docs

[Document(id='b34d8bdb-ef32-44e4-a278-b198508569d3', metadata={'source': 'data/leave_policy.txt'}, page_content='Full-time employees receive 12 sick-leave days per calendar year.\nUnused sick leave is handled according to company policy.\n\nEmployees can request annual leave through the HR portal.\nThe manager must approve the request.'),
 Document(id='1134fa63-c316-4dc5-905b-cee99eb26a40', metadata={'source': 'data/leave_policy.txt'}, page_content='Employees must submit travel reimbursement claims within 30 days\nof completing the business trip.')]

### Generate an answer using retrieved context

#### Create a promtp template 

In [12]:
prompts = ChatPromptTemplate.from_template(
    """
    
    You are an employee-policy assistant.
Answer the question using only the supplied context.
If the context does not contain the answer, say:
"I could not find that information in the available policy documents."
Do not invent policy details.

Context:
{context}

Question:
{question}

Answer:
    
    
    """
)


model = ChatGroq(model="qwen/qwen3.8-27b",temperature=0)

parser = StrOutputParser()


def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata.get('source', 'unknown')} (Section: {doc.metadata.get('section', 'general')})\n{doc.page_content}"
        for doc in docs
    )

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question":RunnablePassthrough()
    }
    | prompts
    | model
    | parser
)


answer = rag_chain.invoke("How many sick-leave days do employees receive?")
print("Answer:\n", answer)

Answer:
 Full-time employees receive 12 sick-leave days per calendar year.
